# Laboratorium 2: Współbieżność i Równoległość w Pythonie
### Skoroszyt Edukacyjny - Wersja dla Studentów

---

## 1. Wstęp: Koncepcja "Wielu Zadań"

Zanim zaczniemy pisać kod, musimy rozróżnić dwa kluczowe pojęcia:

1. **Współbieżność (Concurrency)**: Wykonywanie wielu zadań "na zmianę". Wyobraź sobie kelnera, który obsługuje 5 stolików. Nie robi wszystkiego naraz, ale szybko przełącza się między nimi. Dla klientów wygląda to, jakby obsługiwał ich równocześnie.
2. **Równoległość (Parallelism)**: Wykonywanie wielu zadań faktycznie w tym samym momencie. To sytuacja, w której mamy 5 kelnerów i każdy obsługuje jeden stolik.

W Pythonie współbieżność realizujemy najczęściej za pomocą **Wątków (Threads)**, a równoległość za pomocą **Procesów (Processes)**.

---

## 2. Wielowątkowość (Threading) - Zadania I/O-bound

Wątki są idealne, gdy program większość czasu spędza na **czekaniu** na odpowiedź z sieci (zapytania HTTP). W tym czasie procesor się nudzi – wątki pozwalają mu wysłać kolejne zapytania, nie czekając na poprzednie.

---

### Demo: Scraping Kalendarza Kulturalnego (Krakow.pl)

**Kod zawarty w poniższych komórkach (analogicznie do plików `lab_2_1_demo.py` oraz `lab_2_2_demo.py`) pozwala na pobieranie tytułów wydarzeń kulturalnych z oficjalnego kalendarium miasta Krakowa (krakow.pl).**

Przykładowy adres źródłowy: `https://www.krakow.pl/kalendarium/1919,shw,2026-03-20,0,day.html`.

Demo pokazuje proces pobierania danych z 5 kolejnych stron tego zestawienia:
1. **Wersja sekwencyjna**: Zadanie wykonywane jest krok po kroku, co pozwala zaobserwować sumaryczny czas oczekiwania na każde z zapytań HTTP z osobna (wysoki koszt operacji wejścia/wyjścia).
2. **Optymalizacja**: Kod zostaje zmodyfikowany z użyciem modułu `concurrent.futures`, wykorzystując `ThreadPoolExecutor`.

Dzięki temu zapytania sieciowe są wysyłane równolegle, co drastycznie skraca czas całkowity działania programu, demonstrując praktyczną przewagę wielowątkowości w zadaniach typu **I/O-bound** (zależnych od odpowiedzi sieciowej).

In [ ]:
import requests
from bs4 import BeautifulSoup
import time

def download_site(url):
    """Pobiera jedną stronę i wyciąga tytuły wydarzeń."""
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    event_titles = [item.text.strip() for item in soup.select('.item__link h3')]
    return event_titles

def run_sequential_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie SEKWENCYJNE 5 stron...")
    start = time.time()

    all_titles = []
    for url in sites:
        all_titles.extend(download_site(url))

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania: {time.time() - start:.2f}s")

run_sequential_demo()

In [ ]:
import concurrent.futures

def run_threaded_demo():
    date_str = "2026-03-20"
    base_url = "https://www.krakow.pl/kalendarium/1919,shw"
    sites = [f"{base_url},{date_str},{i},day.html" for i in range(5)]

    print(f"Rozpoczynam pobieranie WIELOWĄTKOWE 5 stron...")
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        results = list(executor.map(download_site, sites))

    all_titles = [title for sublist in results for title in sublist]

    print(f"Pobrano łącznie {len(all_titles)} tytułów.")
    print("Pierwsze 10 wyników:")
    for i, title in enumerate(all_titles[:10], 1):
        print(f"{i}. {title}")

    print(f"\nCzas wykonania (wątki): {time.time() - start:.2f}s")

run_threaded_demo()

---
## 3. Synchronizacja: Problem Hazardu i Lock

Gdy wiele wątków próbuje zmieniać tę samą zmienną w tym samym momencie (np. saldo na koncie), dochodzi do tzw. **Race Condition** (wyścigu). Rozwiązaniem jest **Lock** (blokada).

In [ ]:
import threading

class BankAccount:
    def __init__(self):
        self.balance = 0
        self.lock = threading.Lock()

    def deposit(self, amount):
        with self.lock:
            current = self.balance
            time.sleep(0.0001) # Symulacja opóźnienia
            self.balance = current + amount

account = BankAccount()
with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
    executor.map(lambda _: account.deposit(1), range(100))

print(f"Saldo końcowe: {account.balance} zł (oczekiwano: 100)")

---
## 4. Wieloprocesowość (Multiprocessing) - Zadania CPU-bound

Kiedy musimy wykonać ciężkie obliczenia matematyczne (np. szukanie liczb pierwszych), wątki nam nie pomogą. Musimy użyć osobnych procesów.

**Ważne (macOS/Windows)**: Ze względu na metodę `spawn` startu procesów, funkcje pomocnicze (jak `find_primes`) muszą znajdować się w zewnętrznym pliku `.py` (tutaj: `lab2_functions.py`) i być importowane.

In [ ]:
import multiprocessing
import time
# Importujemy funkcję z oddzielnego pliku, aby uniknąć błędu spawn na macOS
from lab2_functions import find_primes

def run_primes_demo():
    cores = multiprocessing.cpu_count()
    print(f"Praca na {cores} procesach (rdzeniach)...")
    start = time.time()

    limit = 1_000_000
    chunk = limit // cores
    ranges = [(i, i + chunk) for i in range(0, limit, chunk)]

    with multiprocessing.Pool(processes=cores) as pool:
        results = pool.starmap(find_primes, ranges)

    print(f"Zakończono w czasie {time.time() - start:.2f}s.")

if __name__ == "__main__":
    run_primes_demo()

---
# Zadania do samodzielnego wykonania

Poniższe zadania należy zrealizować w oparciu o wiedzę zdobytą na laboratoriach oraz instrukcje zawarte w pliku PDF.

### Zadanie 1 (Threading)
Przy użyciu publicznego API **Cat Facts** (`https://catfact.ninja/fact`), które zwraca przy każdym wywołaniu losowy fakt na temat kotów:
1. Pobierz sekwencyjnie 20 faktów i zmierz czas całkowitego działania programu.
2. Zmodyfikuj kod, aby wysyłać zapytania wielowątkowo przy użyciu `ThreadPoolExecutor`.
3. Porównaj czasy wykonania.

*Podpowiedź: Użyj `requests.get(URL).json().get('fact')`*

In [1]:
import requests
import time
import concurrent.futures

CAT_API_URL = "https://catfact.ninja/fact"
NUM_REQUESTS = 20

def get_cat_fact():
    try:
        response = requests.get(CAT_API_URL, timeout=5)
        response.raise_for_status()
        return response.json().get("fact")
    except Exception as e:
        return f"Błąd: {e}"

def fetch_sequential(n):
    facts = []
    start = time.time()

    for _ in range(n):
        fact = get_cat_fact()
        facts.append(fact)

    duration = time.time() - start
    return facts, duration

def fetch_threaded(n, max_workers=10):
    facts = []
    start = time.time()

    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = executor.map(lambda _: get_cat_fact(), range(n))
        facts = list(results)

    duration = time.time() - start
    return facts, duration

if __name__ == "__main__":
    print("Pobieranie sekwencyjne...")
    seq_facts, seq_time = fetch_sequential(NUM_REQUESTS)

    print("\nPobieranie wielowątkowe...")
    thr_facts, thr_time = fetch_threaded(NUM_REQUESTS)

    print("\n--- WYNIKI ---")
    print(f"Czas sekwencyjny: {seq_time:.2f} s")
    print(f"Czas wielowątkowy: {thr_time:.2f} s")

    print("\nPrzykładowe fakty:")
    for i, fact in enumerate(thr_facts[:5], 1):
        print(f"{i}. {fact}")

Pobieranie sekwencyjne...

Pobieranie wielowątkowe...

--- WYNIKI ---
Czas sekwencyjny: 2.06 s
Czas wielowątkowy: 0.32 s

Przykładowe fakty:
1. It has been scientifically proven that owning cats is good for our health and can decrease the occurrence of high blood pressure and other illnesses.
2. A kitten will typically weigh about 3 ounces at birth.  The typical male housecat will weigh between  7 and 9 pounds, slightly less for female housecats.
3. A cat’s heart beats nearly twice as fast as a human heart, at 110 to 140 beats a minute.
4. A cat’s hearing is better than a dog’s. And a cat can hear high-frequency sounds up to two octaves higher than a human.
5. Cats see six times better in the dark and at night than humans.


### Zadanie 2 (Wątki i Kolejka - Producent-Konsument)
Napisz program o strukturze **producent-consumers**:
1. **Producent**: Generuje kolejne liczby naturalne i dodaje je do kolejki (`queue.Queue`).
2. **Konsument 1**: Pobiera z kolejki tylko liczby **parzyste**.
3. **Konsument 2**: Pobiera z kolejki tylko liczby **nieparzyste**.

Użyj wątków do realizacji producenta i obu konsumentów. Program powinien zakończyć się po przetworzeniu określonej puli liczb.

In [2]:
import queue
import threading
import time

N = 20
q = queue.Queue()
STOP = None

def producent():
    for i in range(N):
        print(f"Producent dodaje: {i}")
        q.put(i)
        time.sleep(0.1)
    q.put(STOP)
    q.put(STOP)

def konsument_parzyste():
    while True:
        item = q.get()
        if item is STOP:
            q.put(STOP)
            break
        if item % 2 == 0:
            print(f"[Parzyste] pobrano: {item}")
        else:
            q.put(item)
        q.task_done()
        time.sleep(0.1)

def konsument_nieparzyste():
    while True:
        item = q.get()
        if item is STOP:
            q.put(STOP)
            break
        if item % 2 == 1:
            print(f"[Nieparzyste] pobrano: {item}")
        else:
            q.put(item)
        q.task_done()
        time.sleep(0.1)

t_prod = threading.Thread(target=producent)
t_k1 = threading.Thread(target=konsument_parzyste)
t_k2 = threading.Thread(target=konsument_nieparzyste)

t_prod.start()
t_k1.start()
t_k2.start()

t_prod.join()
t_k1.join()
t_k2.join()

print("Zakończono program.")

Producent dodaje: 0
[Parzyste] pobrano: 0
Producent dodaje: 1
[Nieparzyste] pobrano: 1
Producent dodaje: 2
[Parzyste] pobrano: 2
Producent dodaje: 3
[Nieparzyste] pobrano: 3
Producent dodaje: 4
[Parzyste] pobrano: 4
Producent dodaje: 5
[Nieparzyste] pobrano: 5
Producent dodaje: 6
[Parzyste] pobrano: 6
Producent dodaje: 7
[Nieparzyste] pobrano: 7
Producent dodaje: 8
[Parzyste] pobrano: 8
Producent dodaje: 9
[Nieparzyste] pobrano: 9
Producent dodaje: 10
[Parzyste] pobrano: 10
Producent dodaje: 11
[Nieparzyste] pobrano: 11
Producent dodaje: 12
[Parzyste] pobrano: 12
Producent dodaje: 13
[Nieparzyste] pobrano: 13
Producent dodaje: 14
[Parzyste] pobrano: 14
Producent dodaje: 15
[Nieparzyste] pobrano: 15
Producent dodaje: 16
[Parzyste] pobrano: 16
Producent dodaje: 17
[Nieparzyste] pobrano: 17
Producent dodaje: 18
[Parzyste] pobrano: 18
Producent dodaje: 19
[Nieparzyste] pobrano: 19
Zakończono program.


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Zadanie 3 (Multiprocessing)
Napisz program, który zrównolegli obliczanie sumy kolejnych stu potęg dla każdej liczby z ciągu liczb naturalnych w dużym zakresie (np. 1 - 10 000).
Użyj modułu `multiprocessing` oraz gotowej funkcji `calculate_power_sum(n)` z pliku `lab2_functions.py`.

Pamiętaj o bezpiecznym uruchamianiu procesów na macOS (`if __name__ == "__main__":`).

In [9]:
import sys
sys.path.append('/content/drive/MyDrive/Colab Notebooks/')
import multiprocessing
import time
from lab2_functions import calculate_power_sum

if __name__ == "__main__":
    start = time.time()

    with multiprocessing.Pool() as pool:
        results = pool.map(calculate_power_sum, range(1, 10001))

    end = time.time()

    print(f"Czas wykonania: {end - start}")

Czas wykonania: 0.7234721183776855
